# Real-Time EEG-to-Text Training Architecture

This notebook implements the real-time EEG-to-Text models based on state-of-the-art Voice-to-Text ASR techniques.

## Models Included:
1. **Conformer (Convolution-augmented Transformer)**: The gold standard for ASR. It uses 1D spatial convolutions for feature extraction, followed by Conformer blocks to capture both local micro-states (CNN) and global sequence intent (Transformer).
2. **CNN-LSTM (DeepSpeech style)**: A classical approach for sequential signal processing.

**Loss Function**: Connectionist Temporal Classification (CTC) for alignment-free training.

In [ ]:
import difflib

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm


def greedy_decoder(logits, char_to_idx):
    idx_to_char = {v: k for k, v in char_to_idx.items()}
    preds = torch.argmax(logits, dim=-1).transpose(0, 1)
    decoded = []
    for seq in preds:
        prev = -1
        text = ""
        for c in seq:
            c = c.item()
            if c != 0 and c != prev:
                text += idx_to_char.get(c, "")
            prev = c
        decoded.append(text)
    return decoded


def calculate_cer(pred_texts, target_texts):
    scores = []
    for p, t in zip(pred_texts, target_texts):
        scores.append(difflib.SequenceMatcher(None, p, t).ratio())
    return sum(scores) / max(1, len(scores))

In [ ]:
# Requirements: pip install torch torchaudio h5py matplotlib
import glob
import os

import h5py
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset

DATASET_PATH = r"C:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\extracted"
MODELS_DIR = "models"
METRICS_DIR = "metrics"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

## 1. Dataset Loading

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, data_dir, max_len=500):
        self.files = glob.glob(os.path.join(data_dir, "**", "*.h5"), recursive=True)
        self.max_len = max_len
        # Simple vocabulary character-level mapping
        self.chars = (
            "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?'-"
        )
        self.char_to_idx = {ch: i + 1 for i, ch in enumerate(self.chars)}
        self.char_to_idx["<PAD>"] = 0
        self.vocab_size = len(self.char_to_idx)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        filename = os.path.basename(file_path)
        transcript = filename.replace(".h5", "").replace("_", " ")

        with h5py.File(file_path, "r") as f:
            keys = list(f.keys())
            if len(keys) > 0:
                eeg_data = f[keys[0]][:]  # shape expected: [105 channels, time_steps]
            else:
                eeg_data = np.zeros((105, self.max_len))

        if len(eeg_data.shape) != 2 or eeg_data.shape[0] != 105:
            eeg_data = np.zeros((105, self.max_len))
        if eeg_data.shape[1] < self.max_len:
            pad_width = self.max_len - eeg_data.shape[1]
            eeg_data = np.pad(eeg_data, ((0, 0), (0, pad_width)), mode="constant")
        else:
            eeg_data = eeg_data[:, : self.max_len]

        # [time_steps, channels]
        eeg_tensor = torch.tensor(eeg_data, dtype=torch.float32).transpose(0, 1)

        target = [self.char_to_idx.get(c, 0) for c in transcript]
        target_tensor = torch.tensor(target, dtype=torch.long)

        subject = (
            os.path.normpath(file_path).split(os.sep)[-3]
            if len(os.path.normpath(file_path).split(os.sep)) >= 3
            else "Unknown"
        )
        return eeg_tensor, target_tensor, subject


dataset = EEGDataset(DATASET_PATH)
print(f"Found {len(dataset)} HDF5 files for training.")

## 2. Models Definition
### 2.1 Conformer (Convolution-augmented Transformer)
State-of-the-art ASR model adapted for EEG spatial decoding.

In [ ]:
from torchaudio.models import Conformer


class ConformerEEG(nn.Module):
    def __init__(self, num_channels=105, input_dim=256, num_classes=65):
        super().__init__()
        # Spatial Convolution to map 105 channels to internal representation
        self.spatial_conv = nn.Conv1d(num_channels, input_dim, kernel_size=3, padding=1)

        # Torchaudio Conformer expects inputs of (batch, time, input_dim)
        self.conformer = Conformer(
            input_dim=input_dim,
            num_heads=4,
            ffn_dim=512,
            num_layers=4,
            depthwise_conv_kernel_size=31,
        )

        # Linear head for character prediction (CTC)
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        # x shape: [batch, time, channels]
        x = x.transpose(1, 2)  # [batch, channels, time]
        x = self.spatial_conv(x)  # [batch, input_dim, time]
        x = x.transpose(1, 2)  # [batch, time, input_dim]

        # Conformer requires input lengths, using max length for simplicity
        lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)

        out, _ = self.conformer(x, lengths)
        out = self.fc(out)  # [batch, time, num_classes]
        return out

### 2.2 DeepSpeech-style (CNN-LSTM)

In [ ]:
class DeepSpeechEEG(nn.Module):
    def __init__(self, num_channels=105, hidden_size=256, num_classes=65):
        super().__init__()
        self.conv1 = nn.Conv1d(num_channels, 64, kernel_size=5, stride=1, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)

        self.lstm = nn.LSTM(
            64, hidden_size, num_layers=2, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = x.transpose(1, 2)

        self.lstm.flatten_parameters()
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

## 3. Training Loop (CTC Loss)

In [ ]:
device = torch.device("cuda")
print(f"Using device: {device}")


def train_model(model, name, dataloader, epochs=5):
    model = model.to(device)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    loss_history = []
    recall_history = []
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    display_handle = display(fig, display_id=True)
    plt.close(fig)
    model.train()
    start_epoch = 0
    checkpoint_path = os.path.join(MODELS_DIR, f"{name}_checkpoint.pth")
    if os.path.exists(checkpoint_path):
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_epoch = checkpoint["epoch"] + 1
            print(f"Resumed {name} training from epoch {start_epoch}")
        except (RuntimeError, EOFError):
            print("Could not load full checkpoint. Starting from scratch.")
    for epoch in range(start_epoch, epochs):
        epoch_loss = 0
        epoch_recall = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")
        for eeg, target, subject in pbar:
            eeg, target = eeg.to(device), target.to(device)
            # Live plotting
            from IPython.display import clear_output

            if pbar.n % 10 == 0:
                clear_output(wait=True)
                _, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
                sample = eeg[0].detach().cpu().numpy()
                for ch in range(min(5, sample.shape[1])):
                    ax1.plot(sample[:, ch] + ch * 2.0)
                ax1.set_title(f"Live Raw EEG Processing (Model: {name})")
                ax1.set_xlabel("Time")
                ax2.plot(loss_history, color="r")
                ax2.set_title(f"CTC Loss (Epoch {epoch + 1}/{epochs})")
                plt.show()
            batch_size = eeg.size(0)
            # Pooling in DeepSpeech halves the time dimension
            out_time = (
                eeg.size(1) // 2 if isinstance(model, DeepSpeechEEG) else eeg.size(1)
            )
            input_lengths = torch.full(
                size=(batch_size,), fill_value=out_time, dtype=torch.long
            )
            target_lengths = torch.tensor([len(target[0])])

            optimizer.zero_grad()
            out = model(eeg)
            out = out.transpose(0, 1)  # [time, batch, classes]
            out = nn.functional.log_softmax(out, dim=2)

            loss = criterion(out, target, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            batch_recall = 0
            if "char_to_idx" in globals() or hasattr(dataset, "char_to_idx"):
                char_to_idx = (
                    dataset.char_to_idx
                    if hasattr(dataset, "char_to_idx")
                    else globals()["char_to_idx"]
                )
                preds = greedy_decoder(out, char_to_idx)
                idx_to_char = {v: k for k, v in char_to_idx.items()}
                targets_text = []
                for t in target:
                    t_list = t.tolist() if hasattr(t, "tolist") else t
                    if isinstance(t_list, int):
                        t_list = [t_list]
                    targets_text.append(
                        "".join([idx_to_char.get(c, "") for c in t_list])
                    )
                batch_recall = calculate_cer(preds, targets_text)
            epoch_recall += batch_recall

            if batch_size > 0 and not hasattr(pbar, "printed_sample"):
                print(
                    f"\n[Comparison] Actual: '{targets_text[0]}' | Predicted: '{preds[0]}'"
                )
                pbar.printed_sample = True

            ax1.clear()
            ax2.clear()
            ax3.clear()
            subj = subject[0] if len(subject) > 0 else "Unknown"

            sample = eeg[0].detach().cpu().numpy()
            for ch in range(min(5, sample.shape[1])):
                ax1.plot(sample[:, ch] + ch * 2.0, label=f"Ch {ch}")
            ax1.legend(loc="upper right")
            ax1.set_title(f"Live Raw EEG (Subj: {subj})")
            ax1.set_xlabel("Time")

            ax2.plot(loss_history + [loss.item()], color="r", label="CTC Loss")
            ax2.set_title("Real-Time CTC Loss")
            ax2.set_xlabel("Batches")
            ax2.legend(loc="upper right")

            ax3.plot(
                recall_history + [batch_recall], color="g", label="Recall (Accuracy)"
            )
            ax3.set_title("Real-Time Character Recall")
            ax3.set_xlabel("Batches")
            ax3.legend(loc="upper right")

            display_handle.update(fig)
            pbar.set_postfix(
                {"Loss": f"{loss.item():.4f}", "Recall": f"{batch_recall:.4f}"}
            )

        avg_loss = epoch_loss / max(1, len(dataloader))
        avg_recall = epoch_recall / max(1, len(dataloader))
        recall_history.append(avg_recall)
        loss_history.append(avg_loss)
        print(f"[{name}] Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss,
        },
        os.path.join(MODELS_DIR, f"{name}_checkpoint.pth"),
    )
    return loss_history


dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

if len(dataset) > 0:
    vocab_size = dataset.vocab_size

    print("Training ConformerEEG...")
    conf_model = ConformerEEG(num_classes=vocab_size)
    conf_losses = train_model(conf_model, "ConformerEEG", dataloader, epochs=5)

    print("Training DeepSpeechEEG...")
    ds_model = DeepSpeechEEG(num_classes=vocab_size)
    ds_losses = train_model(ds_model, "DeepSpeechEEG", dataloader, epochs=5)

    plt.figure(figsize=(10, 5))
    plt.plot(conf_losses, label="Conformer Loss", marker="o")
    plt.plot(ds_losses, label="DeepSpeech Loss", marker="x")
    plt.title("Training Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("CTC Loss")
    plt.legend()

    plot_path = os.path.join(METRICS_DIR, "training_loss_curves.png")
    plt.savefig(plot_path)
    print(f"Saved metrics plot to {plot_path}")
    plt.show()
else:
    print("No HDF5 data found. Check DATASET_PATH.")